In [2]:
import struct
import socket

In [3]:

def build_dns_query(domain_name, query_type=1):
    """
    Constrói uma mensagem de consulta DNS (A record) conforme RFC 1035.
    """
    # 1. Cabeçalho (Header)
    # ID: Random, Flags: 0x0100 (Recursion Desired), QDCOUNT: 1, ANCOUNT: 0, NSCOUNT: 0, ARCOUNT: 0
    transaction_id = 0xAAAA # Exemplo
    flags = 0x0100
    qdcount = 1
    ancount = 0
    nscount = 0
    arcount = 0
    
    # '!' = network order (big-endian), H = unsigned short (2 bytes)
    header = struct.pack('!HHHHHH', transaction_id, flags, qdcount, ancount, nscount, arcount)

    # 2. Pergunta (Question) - Nome codificado (ex: www.google.com -> \x03www\x06google\x03com\x00)
    encoded_name = b""
    for part in domain_name.encode("ascii").split(b"."):
        encoded_name += bytes([len(part)]) + part
    encoded_name += b"\x00"  # Finalizador

    # Tipo: A (1) | Classe: IN (1)
    qtype = query_type
    qclass = 1 # IN
    
    question = encoded_name + struct.pack('!HH', qtype, qclass)

    return header + question

# --- Uso ---
domain = "google.com"
dns_message = build_dns_query(domain)

print(f"Mensagem DNS para {domain} ({len(dns_message)} bytes):")
print(dns_message)

# Opcional: Enviar para um servidor DNS (ex: 8.8.8.8)
# sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
# sock.sendto(dns_message, ('8.8.8.8', 53))
# data, _ = sock.recvfrom(512)


Mensagem DNS para google.com (28 bytes):
b'\xaa\xaa\x01\x00\x00\x01\x00\x00\x00\x00\x00\x00\x06google\x03com\x00\x00\x01\x00\x01'


In [ ]:
dns_query = dns_message

DNS_SERVER = "8.8.8.8"
PORT = 53

# 3. Criar socket UDP (SOCK_DGRAM)
# Note: Raw sockets (SOCK_RAW) costumam exigir privilégios de root/admin,
# mas SOCK_DGRAM envia o payload DNS "cru" dentro de um UDP gerado pelo SO.
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

try:
    print(f"Enviando query para {DNS_SERVER}...")
    
    # 4. Enviar a mensagem pronta
    sock.sendto(dns_query, (DNS_SERVER, PORT))
    
    # 5. Receber resposta
    data, addr = sock.recvfrom(1024)
    print(f"Resposta recebida de {addr}:")
    print(data.hex()) # Exibe a resposta em hexadecimal

finally:
    sock.close()
